In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from geopy.distance import geodesic
import geopandas as gpd
import pulp
import folium
import osmnx as ox
import folium
from streamlit_folium import st_folium
from geopy.geocoders import Nominatim
import time
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderUnavailable

import ssl
import certifi


In [ ]:

ctx = ssl.create_default_context(cafile=certifi.where())
geolocator = Nominatim(user_agent="geoapi", ssl_context=ctx)
def reverse_geocode(lat, lon, retries=5):
    for i in range(retries):
        try:
            location = geolocator.reverse((lat, lon), exactly_one=True)
            return location.address if location else None
        except (GeocoderTimedOut, GeocoderUnavailable):
            if i < retries - 1:
                time.sleep(2)
                continue
            else:
                return None


import and cleaning

In [3]:
cam_locations = pd.read_csv(r'Automated_Traffic_Enforcement_Table.csv')

crash_data = pd.read_csv(r'Crashes_in_DC-2.csv')

/var/folders/5v/vp3xzrf13wd2c5mb7v92nnj40000gn/T/ipykernel_18907/3357910157.py:3: DtypeWarning: Columns (3,18) have mixed types. Specify dtype option on import or set low_memory=False.
  crash_data = pd.read_csv(r'Crashes_in_DC-2.csv')


In [4]:
crash_data=crash_data[crash_data['SPEEDING_INVOLVED']>=1]

In [5]:
crash_data['REPORTDATE'] = pd.to_datetime(crash_data['REPORTDATE'])
crash_data_filtered = crash_data[crash_data['REPORTDATE'].between('2010-01-01','2025-10-17',inclusive='both')]
crash_data_filtered = crash_data_filtered.rename({"LATITUDE":"latitude",'LONGITUDE':'longitude'},axis=1)

In [6]:
cam_locations['START_DATE']= pd.to_datetime(cam_locations['START_DATE'])
activecam_locations=cam_locations[cam_locations['ACTIVE_STATUS']!='Retired'].reset_index()

active_speed_cam_locations = activecam_locations[activecam_locations['ENFORCEMENT_TYPE']=='Speed']
cam_data_w_coordinates = active_speed_cam_locations[active_speed_cam_locations['CAMERA_LATITUDE'].notna()]

In [7]:
crash_data_filtered = crash_data_filtered.rename({"LATITUDE":"latitude",'LONGITUDE':'longitude'},axis=1)

In [ ]:
crash_data_filtered[crash_data_filtered['']]

,X,Y,CRIMEID,CCN,REPORTDATE,ROUTEID,MEASURE,OFFSET,STREETSEGID,ROADWAYSEGID,...,BLOCKKEY,SUBBLOCKKEY,CORRIDORID,NEARESTINTKEY,MAJORINJURIESOTHER,MINORINJURIESOTHER,UNKNOWNINJURIESOTHER,FATALOTHER,OBJECTID,cluster_id
25,-8.569242e+06,4.707984e+06,23510896,10165388,2010-11-14 11:30:00+00:00,12053882,138.96,0.07,9508.0,10231.0,...,db1a312e9c98ff2d1708cb786c7f0e40,db1a312e9c98ff2d1708cb786c7f0e40,12053882_1,e06ca507a64e6051e94ccd0a80c6fc3d,NaN,NaN,NaN,NaN,397404295,0
26,-8.566655e+06,4.709783e+06,23511254,10165664,2010-11-15 01:57:00+00:00,12003632,189.37,0.01,10969.0,4569.0,...,5fb744af73b61ded65586426854b952c,9d44b1de313d48f9c85ebdb73b1c96f0,Blockkey Not Found on Corridor,33c364fc59975fc91314be9491356b28,NaN,NaN,NaN,NaN,397404296,1
38,-8.574057e+06,4.716823e+06,23472237,10146224,2010-10-08 23:50:00+00:00,11079882,987.11,17.95,-9.0,27437.0,...,c2785e5703a0ed0a2f0c251eab0cd293,0795229770c26f9022e81a82260c40d6,11079882_2,5168a56829b97fb0055df38efd9d106d,NaN,NaN,NaN,NaN,397404308,-1
49,-8.571346e+06,4.699229e+06,23480471,10150293,2010-10-16 04:00:00+00:00,13009362,201.00,24.56,2629.0,16431.0,...,6e19196bd62d0f3be4e9e1e13344d23d,6e19196bd62d0f3be4e9e1e13344d23d,13009362_1,d124769cadd7594e5a9292a6e1e57a71,NaN,NaN,NaN,NaN,397404319,2
56,-8.572540e+06,4.711065e+06,23487371,10153748,2010-10-23 02:32:00+00:00,12040432,0.00,22.04,-9.0,25425.0,...,2b5adf5e073abf3708a3fae53f00486b,7b8416bc2858181edec642be80be8ca2,12040432_1,ad9fe99047ed2f7d8674d48c5866b586,NaN,NaN,NaN,NaN,397404326,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
337988,-8.575295e+06,4.717700e+06,66694224806,25157058,2025-10-15 05:04:00+00:00,11011572,1647.52,32.48,NaN,NaN,...,8eecf23c4ee305f8220f42ac3bbef54f,8338a1557c53e9ceb701f24b49f45e07,NaN,93998b1d2463c66309d3f82ec2888593,0.0,0.0,0.0,0.0,397742258,-1
338032,-8.578698e+06,4.713738e+06,66736862616,25157685,2025-10-16 15:35:00+00:00,11025152,5301.35,24.95,NaN,NaN,...,5f11eb330d4aa9e5e2ccf76a0fb05d31,5f11eb330d4aa9e5e2ccf76a0fb05d31,11025152_2,79bee7bbf3d7f75fd63c247457d09634,0.0,0.0,0.0,0.0,397742302,-1
338048,-8.564422e+06,4.709181e+06,66666009488,25155975,2025-10-13 00:04:00+00:00,Route not found,0.00,0.00,NaN,NaN,...,Route not found,Route not found,Route not found,Route not found,0.0,0.0,0.0,0.0,397742318,11
338121,-8.567671e+06,4.705899e+06,66691824523,25156904,2025-10-14 22:00:00+00:00,15031912,2811.04,5.03,NaN,NaN,...,a2991740c6a9b59165c476e91531f279,7aa82a612813814b27c62203a3da8def,15031912_3,f5c461f2e8cc41b085cef4841bf862b1,0.0,0.0,0.0,0.0,397742391,-1


crash clustering

In [8]:
def cluster_crashes(df,min_sample, eps_meters):
    
    coords = df[['latitude', 'longitude']].to_numpy()
    
    eps_km = eps_meters / 1000
    kms_per_radian = 6371.0088
    eps = eps_km / kms_per_radian

    db = DBSCAN(eps=eps, min_samples=min_sample, metric='haversine').fit(np.radians(coords))
    
    df['cluster_id'] = db.labels_
    
    return df

In [198]:
cluster_size_list = [5,10,15,20,25,30,35]
distance_list = [80,160,240,320]

In [199]:
res_dict = {}
for c in cluster_size_list:
    for d in distance_list:
        crash_clusters=cluster_crashes(crash_data_filtered,c,eps_meters=d)
        cluster_sizes = crash_clusters.groupby('cluster_id',as_index=False).size().rename({'size':"crash_count"},axis=1)
        num_clusters = cluster_sizes.shape[0]
        res_dict[f"{c}-{d}"] = num_clusters-1


In [201]:
records = []
for key, val in res_dict.items():
    c, d = key.split('-')
    records.append((int(c), float(d), val))

res_df = pd.DataFrame(records, columns=['c', 'd', 'num_clusters_minus1'])

# Pivot so c is index, d is column, values are num_clusters_minus1
res_pivot = res_df.pivot(index='c', columns='d', values='num_clusters_minus1')

res_pivot

d,80.0,160.0,240.0,320.0
c,,,,
5,312,163,50,13
10,76,116,40,11
15,22,66,53,14
20,9,41,45,20
25,4,25,41,23
30,0,16,34,27
35,0,6,24,28


In [202]:
res_pivot.reset_index(drop=False)

d,c,80.0,160.0,240.0,320.0
0,5,312,163,50,13
1,10,76,116,40,11
2,15,22,66,53,14
3,20,9,41,45,20
4,25,4,25,41,23
5,30,0,16,34,27
6,35,0,6,24,28


In [194]:
#used 80m for ~standard city block size 
crash_clusters=cluster_crashes(crash_data_filtered,35,eps_meters=80)

In [195]:
#crash density per cluster
cluster_sizes = crash_clusters.groupby('cluster_id',as_index=False).size().rename({'size':"crash_count"},axis=1)

In [196]:
cluster_sizes

,cluster_id,crash_count
0,-1,6645


In [11]:
crash_geo_df = gpd.GeoDataFrame(crash_clusters,geometry=gpd.points_from_xy(crash_clusters['longitude'],crash_clusters['latitude']))

In [12]:
#getting central point for each cluster
cluster_centroids = crash_geo_df[crash_geo_df['cluster_id'] != -1].dissolve(by='cluster_id').centroid

crash_cluster_centroids= cluster_centroids.reset_index(drop=False)
crash_cluster_centroids.columns = ['cluster_id','crash_centroid_coords'] 


In [13]:
crash_cluster_centroids['cluster_id'].duplicated().sum()

np.int64(0)

In [14]:

crash_geo_df = gpd.GeoDataFrame(
    crash_cluster_centroids,               
    geometry='crash_centroid_coords',          
    crs="EPSG:4326"                
)

In [111]:
crash_geo_df

,cluster_id,crash_centroid_coords
0,0,POINT (-76.95547 38.91697)
1,1,POINT (-77.00935 38.90727)
2,2,POINT (-76.94604 38.90078)
3,3,POINT (-76.97275 38.87436)
4,4,POINT (-76.9427 38.89256)
...,...,...
71,71,POINT (-76.95903 38.88995)
72,72,POINT (-77.01415 38.90543)
73,73,POINT (-76.94802 38.88984)
74,74,POINT (-77.0144 38.88205)


In [15]:
#getting geodf for cam locations
cam_locatios_geo = gpd.GeoDataFrame(cam_data_w_coordinates,geometry=gpd.points_from_xy(cam_data_w_coordinates['CAMERA_LONGITUDE'],cam_data_w_coordinates['CAMERA_LATITUDE']),crs="EPSG:4326")[['ENFORCEMENT_SPACE_CODE','geometry']]

In [16]:
cam_locations_proj = cam_locatios_geo.to_crs(epsg=26918)
crash_locations_proj = crash_geo_df.to_crs(epsg=26918)

In [112]:
crash_locations_proj

,cluster_id,crash_centroid_coords
0,0,POINT (330467.686 4309380.266)
1,1,POINT (325772.665 4308405.808)
2,2,POINT (331246.682 4307565.772)
3,3,POINT (328867.1 4304683.49)
4,4,POINT (331516.991 4306648.162)
...,...,...
71,71,POINT (330094.628 4306388.113)
72,72,POINT (325351.348 4308210.353)
73,73,POINT (331048.992 4306355.392)
74,74,POINT (325272.306 4305615.716)


In [17]:
#getting nearest cam to crash clusters
nearest_cameras = gpd.sjoin_nearest(crash_locations_proj, cam_locations_proj, distance_col="distance_m")


In [18]:
nearest_cameras = nearest_cameras.drop_duplicates(subset='cluster_id')

In [19]:
nearest_cameras = (nearest_cameras.sort_values(by="distance_m", ascending=True).drop_duplicates(subset="cluster_id", keep="first"))

In [20]:
#merging and renaming
neartest_cam_to_crash_clusters = nearest_cameras.merge(cluster_sizes,on='cluster_id',how='left')
neartest_cam_to_crash_clusters=neartest_cam_to_crash_clusters.rename({'crash_centroid_coords':'crash_centroid_coords_projection'},axis=1)
neartest_cam_to_crash_clusters=neartest_cam_to_crash_clusters.merge(cam_locatios_geo,on='ENFORCEMENT_SPACE_CODE',how='left').rename({'geometry':"camera_coordinates"},axis=1)
neartest_cam_to_crash_clusters= neartest_cam_to_crash_clusters.merge(crash_cluster_centroids,on='cluster_id',how='left')
neartest_cam_to_crash_clusters= neartest_cam_to_crash_clusters.rename({"crash_centroid_coords":'crash_centroid_coords_coordinates','ENFORCEMENT_SPACE_CODE':"cam_name",'distance_m':'nearst_camera_distance'},axis=1).drop('index_right',axis=1)

Recomendation Model

In [21]:
neartest_cam_to_crash_clusters['camera_coordinates_longitude']=neartest_cam_to_crash_clusters['camera_coordinates'].x
neartest_cam_to_crash_clusters['camera_coordinates_latitude']=neartest_cam_to_crash_clusters['camera_coordinates'].y

neartest_cam_to_crash_clusters['crash_centroid_longitude']=neartest_cam_to_crash_clusters['crash_centroid_coords_coordinates'].x
neartest_cam_to_crash_clusters['crash_centroid_latitude']=neartest_cam_to_crash_clusters['crash_centroid_coords_coordinates'].y

In [22]:
#filtering for proximity to current camera locations
neartest_cam_to_crash_clusters_over_eighty_meters = neartest_cam_to_crash_clusters[neartest_cam_to_crash_clusters['nearst_camera_distance']>80]

In [23]:
neartest_cam_to_crash_clusters_over_eighty_meters=neartest_cam_to_crash_clusters_over_eighty_meters.set_index('cluster_id')

In [ ]:
#calculating distance between crash clusters, running slow room to improve
crash_sites = neartest_cam_to_crash_clusters_over_eighty_meters[['crash_centroid_latitude', 'crash_centroid_longitude']]
# crash_sites['crash_cluster_address']= crash_sites.apply(lambda x: reverse_geocode(x['crash_centroid_latitude'],x['crash_centroid_longitude']),axis=1)
# distance = pd.DataFrame([
#     [
#         geodesic((crash_sites.loc[i, 'crash_centroid_latitude'], crash_sites.loc[i, 'crash_centroid_longitude']),
#                  (crash_sites.loc[j, 'crash_centroid_latitude'], crash_sites.loc[j, 'crash_centroid_longitude'])).meters
#         for j in crash_sites.index
#     ]
#     for i in crash_sites.index
# ], columns=crash_sites.index, index=crash_sites.index)

/var/folders/5v/vp3xzrf13wd2c5mb7v92nnj40000gn/T/ipykernel_18907/1828373236.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  crash_sites['crash_cluster_address']= crash_sites.apply(lambda x: reverse_geocode(x['crash_centroid_latitude'],x['crash_centroid_longitude']),axis=1)


In [204]:
crash_sites

,crash_centroid_latitude,crash_centroid_longitude,crash_cluster_address,crash_count
cluster_id,,,,
1,38.907273,-77.009347,"3, New York Avenue Northwest, Ward 5, Washingt...",30
39,38.872376,-76.968961,"1501, 27th Street Southeast, Randle Highlands,...",10
8,38.862026,-76.997823,"Firth Sterling Avenue Southeast, Barry Farm, W...",14
7,38.911473,-76.934518,"Supreme Learning Center, 1615, Kenilworth Aven...",21
27,38.850804,-76.973972,"23rd Street Southeast, Buena Vista, Ward 8, Wa...",13
...,...,...,...,...
55,38.882277,-77.017928,"3rd Street Tunnel, Ward 6, Washington, Distric...",25
45,38.882091,-77.022044,"601, 7th Street Southwest, Southwest Waterfron...",10
48,38.882251,-77.025772,"Southwest Freeway, Ward 6, Washington, Distric...",13


In [25]:
# from geopy.distance import geodesic



# camera_sites = neartest_cam_to_crash_clusters[['cam_name', 'camera_coordinates_latitude', 'camera_coordinates_longitude']].drop_duplicates().reset_index(drop=True)

# distance = pd.DataFrame([
#     [
#         geodesic((neartest_cam_to_crash_clusters.loc[i, 'crash_centroid_latitude'], neartest_cam_to_crash_clusters.loc[i, 'crash_centroid_longitude']),
#                  (camera_sites.loc[j, 'camera_coordinates_latitude'], camera_sites.loc[j, 'camera_coordinates_longitude'])).meters
#         for j in range(len(camera_sites))
#     ]
#     for i in range(len(neartest_cam_to_crash_clusters))
# ], columns=camera_sites['cam_name'], index=neartest_cam_to_crash_clusters['cluster_id'])

In [160]:
# setting number of new camera needed
new_cams_needed = 1

crash_count_dict = neartest_cam_to_crash_clusters_over_eighty_meters['crash_count'].to_dict()

In [161]:


#initiate lp 
rec_model = pulp.LpProblem('cam_placement_recs',pulp.LpMinimize)

x_matrix = pulp.LpVariable.dicts('assign',(distance.index,distance.columns),0,1,cat='Binary')
y = pulp.LpVariable.dicts('open',distance.columns,0,1,cat='binary')


# equation multiplying crash density by distance from other crash cameras to make sure new camera locations arent stacked
rec_model += pulp.lpSum([(crash_count_dict[i]*2)*distance.loc[i,j]*x_matrix[i][j] for i in distance.index for j in distance.columns])



#contrains only one cam per cluster
for i in distance.index:
    rec_model += pulp.lpSum([x_matrix[i][j] for j in distance.columns])==1

for i in distance.index:
    for j in distance.columns:
        rec_model += x_matrix[i][j] <= y[j]


#contrains camera recommendations to input cameras needed
rec_model += pulp.lpSum([y[j] for j in distance.columns]) == new_cams_needed

rec_model.solve(pulp.PULP_CBC_CMD(msg=False))


1

In [162]:
#outputs
cluster_id_recomendations = [j for j in distance.columns if y[j].value() == 1]
assignments = {i: [j for j in distance.columns if x_matrix[i][j].value() == 1][0] for i in distance.index}


In [163]:
neartest_cam_to_crash_clusters_over_eighty_meters[neartest_cam_to_crash_clusters_over_eighty_meters['crash_count']==neartest_cam_to_crash_clusters_over_eighty_meters['crash_count'].max()]

,crash_centroid_coords_projection,cam_name,nearst_camera_distance,crash_count,camera_coordinates,crash_centroid_coords_coordinates,camera_coordinates_longitude,camera_coordinates_latitude,crash_centroid_longitude,crash_centroid_latitude
cluster_id,,,,,,,,,,
3,POINT (328867.1 4304683.49),ATE 0532,235.944296,39,POINT (-76.97043 38.87325),POINT (-76.97275 38.87436),-76.97043,38.87325,-76.972751,38.874357


In [164]:
neartest_cam_to_crash_clusters_over_eighty_meters.sort_values('crash_count')

,crash_centroid_coords_projection,cam_name,nearst_camera_distance,crash_count,camera_coordinates,crash_centroid_coords_coordinates,camera_coordinates_longitude,camera_coordinates_latitude,crash_centroid_longitude,crash_centroid_latitude
cluster_id,,,,,,,,,,
64,POINT (325787.459 4301504.236),ATE 0217,528.797542,10,POINT (-77.00356 38.84144),POINT (-77.00743 38.84512),-77.003563,38.841437,-77.007426,38.845120
73,POINT (331048.992 4306355.392),ATE 0264,240.668649,10,POINT (-76.94525 38.8898),POINT (-76.94802 38.88984),-76.945250,38.889800,-76.948024,38.889837
50,POINT (325334.546 4307629.015),ATE 0257,277.238008,10,POINT (-77.01486 38.90263),POINT (-77.0142 38.90019),-77.014855,38.902634,-77.014199,38.900190
6,POINT (330520.895 4309528.4),ATE 0144,290.093663,10,POINT (-76.95756 38.91989),POINT (-76.95489 38.91831),-76.957560,38.919890,-76.954893,38.918312
75,POINT (326715.193 4301516.916),ATE 0458,293.021275,10,POINT (-77.00008 38.84501),POINT (-76.99675 38.84542),-77.000080,38.845010,-76.996745,38.845417
...,...,...,...,...,...,...,...,...,...,...
33,POINT (324675.491 4309454.823),ATE 0349,179.960757,28,POINT (-77.02053 38.9156),POINT (-77.02226 38.9165),-77.020533,38.915604,-77.022261,38.916502
1,POINT (325772.665 4308405.808),ATE 0478,83.034243,30,POINT (-77.00852 38.90765),POINT (-77.00935 38.90727),-77.008520,38.907650,-77.009347,38.907273
31,POINT (329022.447 4309524.09),ATE 0154,779.994691,32,POINT (-76.97912 38.92244),POINT (-76.97217 38.91798),-76.979120,38.922440,-76.972167,38.917983


In [165]:
cluster_id_recomendations



[16]

In [166]:
for e in cluster_id_recomendations:
    print(e,crash_count_dict[e])

16 17


In [150]:
crash_count_dict[46],crash_count_dict[53]

(12, 24)

In [ ]:
crash_count_dict[11],crash_count_dict[50],crash_count_dict[63],

(23, 10, 13)

Sample Viz

In [118]:
gdf = ox.geocode_to_gdf({'city': 'Washington D.C.'})

In [80]:
cam_locs_coord_only = cam_locatios_geo[['geometry']]

cam_locs_coord_only['source']='cams'

In [81]:
crash_data_viz=crash_geo_df.merge(cluster_sizes,on='cluster_id',how='left')
crash_locs_coords_only = crash_data_viz[['crash_centroid_coords','crash_count']]

crash_locs_coords_only['source']='crash'

crash_locs_coords_only = crash_locs_coords_only.rename({'crash_centroid_coords':'geometry'},axis=1)

In [82]:
master_coords = gpd.GeoDataFrame(pd.concat([cam_locs_coord_only,crash_locs_coords_only],axis=0), crs="EPSG:4326")

master_coords['radius_scaled'] = master_coords['crash_count'].fillna(1)
master_coords['radius_scaled'] = master_coords['radius_scaled']*3 * 5 

In [83]:
new_cams = neartest_cam_to_crash_clusters_over_eighty_meters.loc[cluster_id_recomendations][['crash_centroid_coords_coordinates']]

In [108]:
new_cams['crash_count'] = new_cams.index.map(crash_count_dict)

In [84]:
new_cams= new_cams.join(crash_sites['crash_cluster_address'],how='left')

In [110]:
new_cams

,geometry,crash_cluster_address,source,crash_count
cluster_id,,,,
1,POINT (-77.00935 38.90727),"3, New York Avenue Northwest, Ward 5, Washingt...",new_cam,30
11,POINT (-76.94888 38.89511),"Benning Road Northeast, Ward 7, Washington, Di...",new_cam,23
43,POINT (-76.99363 38.83917),"3701, Wheeler Road Southeast, Congress Heights...",new_cam,10
66,POINT (-76.94972 38.8618),"3810, Southern Avenue Southeast, Ward 7, Washi...",new_cam,11
51,POINT (-77.00948 38.95498),"5, Missouri Avenue Northwest, Manor Park, Fort...",new_cam,10
3,POINT (-76.97275 38.87436),"2323, Pennsylvania Avenue Southeast, Ward 7, W...",new_cam,39
0,POINT (-76.95547 38.91697),"Washington Times, 3600, New York Avenue Northe...",new_cam,25
5,POINT (-76.97549 38.89877),"725, 20th Street Northeast, Kingman Park, Carv...",new_cam,38
54,POINT (-76.99009 38.86771),"Anacostia Freeway, Ward 8, Washington, Distric...",new_cam,14


In [ ]:
new_cams['source'] = 'new_cam'

new_cams = new_cams.rename({'crash_centroid_coords_coordinates':'geometry'},axis=1)

In [87]:
master_coords=pd.concat([master_coords,new_cams])

/Users/willbrennan/Desktop/Coding/school_repo/school_code/CSE 6242/myenv/lib/python3.10/site-packages/geopandas/array.py:1755: UserWarning: CRS not set for some of the concatenation inputs. Setting output's CRS as WGS 84 (the single non-null crs provided).
  return GeometryArray(data, crs=_get_common_crs(to_concat))


In [70]:
maps = gdf.explore(name="DC", height=600, width=800)

In [89]:
#red = camera

maps = master_coords.explore(
    m=maps,
    column='source',
    cmap=['red','blue','black'],
    legend=True,
    marker_type='circle',
    radius='radius_scaled',  
    tooltips=['crash_count']
)

In [90]:
folium.LayerControl().add_to(maps)

maps.save("map.html")

- get closest camera location to each crash grouping
- leaflet uses geojson, structure output accordingly

